In [ ]:
# -*- coding: utf-8 -*-
"""ASL_Sign_Language_Training_v2.ipynb

Automatically generated by Colab.

# 🤟 ASL Sign Language Recognition — Full Training Pipeline v2

**IMPROVEMENTS:**
- Removes black/empty skeleton images before training
- Checks class balance
- Two-phase training (frozen base + fine-tuning)
- Confusion matrix + classification report
- Better callbacks

**Steps:**
1. Setup GPU + install packages
2. Download ASL dataset from Kaggle
3. Download MediaPipe hand landmarker model
4. Pre-generate skeleton dataset (runs MediaPipe **once**, saves to disk)
5. **Clean up black/empty images** ← NEW
6. **Check class balance** ← NEW
7. Verify skeletons look correct (visual check)
8. Train MobileNetV2 on skeleton dataset (2-phase) ← IMPROVED
9. Evaluate with confusion matrix ← NEW
10. Test the trained model

---
### ⚙️ TEST MODE vs FULL MODE
"""

# ════════════════════════════════════════════════════
# ⚙️  CHANGE ONLY THIS CELL
# ════════════════════════════════════════════════════

TEST_MODE = False   # ← True = quick test (5 min) | False = full training (30 min)

if TEST_MODE:
    # ── Quick smoke-test settings ───────────────────
    BATCH_SIZE       = 8
    EPOCHS_PHASE1    = 2
    EPOCHS_PHASE2    = 2
    MAX_CLASSES      = 5      # only use first 5 letter classes (A-E)
    MAX_IMGS_PER_CLS = 50     # only 50 images per class
    print('🧪 TEST MODE  — batch=8, epochs=2+2, 5 classes, 50 imgs/class')
    print('   Expected time: ~5-7 min on Colab GPU')
    print('   Switch TEST_MODE=False when this works correctly.')
else:
    # ── Full training settings ──────────────────────
    BATCH_SIZE       = 32
    EPOCHS_PHASE1    = 15     # frozen base
    EPOCHS_PHASE2    = 20     # fine-tuning
    MAX_CLASSES      = None   # all 29 classes
    MAX_IMGS_PER_CLS = None   # all images (~3000 per class)
    print('🚀 FULL MODE  — batch=32, epochs=15+20, all 29 classes, all images')
    print('   Expected time: ~35-45 min on Colab T4 GPU')

# Fixed config (don't change)
IMG_SIZE        = (224, 224)
SKELETON_DIR    = '/content/skeleton_dataset'
MODEL_SAVE_PATH = '/content/sign_language_model_best.keras'
MODEL_FINAL_PATH = '/content/sign_language_model_final.keras'
CLASS_JSON_PATH = '/content/class_names.json'
TASK_FILENAME   = '/content/hand_landmarker.task'
TASK_URL        = ('https://storage.googleapis.com/mediapipe-models/'
                   'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task')

"""## Cell 2 — Check GPU"""

# Verify GPU is enabled
!nvidia-smi
print()

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ GPU ready: {gpus}')
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → T4 GPU')

"""## Cell 3 — Install packages"""

!pip install -q mediapipe kagglehub scikit-learn seaborn

import mediapipe as mp
import kagglehub
import cv2
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print(f'✅ mediapipe  {mp.__version__}')
print(f'✅ opencv     {cv2.__version__}')
print(f'✅ numpy      {np.__version__}')
print(f'✅ tensorflow {tf.__version__}')

"""## Cell 4 — Setup Kaggle credentials"""

from google.colab import files

print('Upload your kaggle.json file:')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('✅ Kaggle credentials saved')

"""## Cell 5 — Download dataset + MediaPipe model"""

import urllib.request

# Download ASL Alphabet dataset
print('⬇️  Downloading ASL Alphabet dataset...')
base_path = kagglehub.dataset_download('grassknoted/asl-alphabet')

RAW_DIR = os.path.join(base_path, 'asl_alphabet_train', 'asl_alphabet_train')
if not os.path.exists(RAW_DIR):
    RAW_DIR = os.path.join(base_path, 'asl_alphabet_train')

all_classes = sorted([d for d in os.listdir(RAW_DIR)
                       if os.path.isdir(os.path.join(RAW_DIR, d))])
print(f'✅ Found {len(all_classes)} classes: {all_classes}')
print(f'   Path: {RAW_DIR}')

# Download MediaPipe hand landmarker
if not os.path.exists(TASK_FILENAME):
    print(f'\n⬇️  Downloading hand_landmarker.task...')
    urllib.request.urlretrieve(TASK_URL, TASK_FILENAME)
    size_mb = os.path.getsize(TASK_FILENAME) / 1e6
    print(f'✅ Downloaded ({size_mb:.1f} MB)')
else:
    print(f'\n✅ hand_landmarker.task already exists')

"""## Cell 6 — Core functions (skeleton pipeline)"""

from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision as mp_vision

HAND_CONNECTIONS = [
    (c.start, c.end)
    for c in mp_vision.HandLandmarksConnections.HAND_CONNECTIONS
]

def to_pixel(val: float, min_val: float, size: float, canvas: int = 224) -> int:
    return max(0, min(canvas - 1, int(((val - min_val) / size) * canvas)))

def build_detector():
    base_options = mp_tasks.BaseOptions(model_asset_path=TASK_FILENAME)
    options = mp_vision.HandLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.3,
        min_hand_presence_confidence=0.3,
        min_tracking_confidence=0.3,
    )
    return mp_vision.HandLandmarker.create_from_options(options)

def image_to_skeleton(img_bgr: np.ndarray, detector) -> tuple:
    canvas = np.zeros((224, 224, 3), dtype=np.uint8)
    img_rgb = cv2.cvtColor(cv2.resize(img_bgr, (224, 224)), cv2.COLOR_BGR2RGB)
    img_uint8 = np.ascontiguousarray(img_rgb)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_uint8)
    result = detector.detect(mp_image)

    if not result.hand_landmarks:
        return canvas, False

    lms = result.hand_landmarks[0]
    xs = [lm.x for lm in lms]; ys = [lm.y for lm in lms]
    x0, x1 = min(xs), max(xs); y0, y1 = min(ys), max(ys)

    pad_x = (x1 - x0) * 0.15; pad_y = (y1 - y0) * 0.15
    x0 -= pad_x; x1 += pad_x; y0 -= pad_y; y1 += pad_y
    w = max(x1 - x0, 0.01); h = max(y1 - y0, 0.01)

    for s, e in HAND_CONNECTIONS:
        p1, p2 = lms[s], lms[e]
        cv2.line(canvas,
                 (to_pixel(p1.x, x0, w), to_pixel(p1.y, y0, h)),
                 (to_pixel(p2.x, x0, w), to_pixel(p2.y, y0, h)),
                 (255, 255, 255), 2)
    for lm in lms:
        cv2.circle(canvas,
                   (to_pixel(lm.x, x0, w), to_pixel(lm.y, y0, h)),
                   4, (220, 220, 220), -1)
    return canvas, True

print('✅ Core functions defined')

"""## Cell 7 — Pre-generate skeleton dataset"""

def generate_skeleton_dataset(raw_dir, out_dir, max_classes=None, max_per_class=None):
    classes = sorted([d for d in os.listdir(raw_dir)
                       if os.path.isdir(os.path.join(raw_dir, d))])
    if max_classes:
        classes = classes[:max_classes]

    total = 0
    for cls in classes:
        imgs = [f for f in os.listdir(os.path.join(raw_dir, cls))
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if max_per_class:
            imgs = imgs[:max_per_class]
        total += len(imgs)

    mode_str = f'{len(classes)} classes × up to {max_per_class or "all"} imgs'
    print(f'Generating skeletons: {mode_str} = {total} total images')
    print(f'Output: {out_dir}\n')

    detector = build_detector()
    processed = 0
    skipped = 0
    no_hand = 0

    for cls in classes:
        src_dir = os.path.join(raw_dir, cls)
        dst_dir = os.path.join(out_dir, cls)
        os.makedirs(dst_dir, exist_ok=True)

        imgs = [f for f in os.listdir(src_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        if max_per_class:
            imgs = imgs[:max_per_class]

        for fname in imgs:
            dst_path = os.path.join(dst_dir, fname)
            if os.path.exists(dst_path):
                skipped += 1; processed += 1
                continue

            bgr = cv2.imread(os.path.join(src_dir, fname))
            if bgr is None:
                processed += 1; continue

            skeleton, found = image_to_skeleton(bgr, detector)
            if not found:
                no_hand += 1

            cv2.imwrite(dst_path, cv2.cvtColor(skeleton, cv2.COLOR_RGB2BGR))
            processed += 1

            if processed % 200 == 0:
                pct = processed / total * 100
                print(f'  [{processed:5}/{total}] {pct:5.1f}%  '
                      f'no_hand={no_hand}  skipped={skipped}')

    new_imgs = processed - skipped
    found_rate = (new_imgs - no_hand) / max(new_imgs, 1) * 100
    print(f'\n✅ Done!')
    print(f'   New images processed : {new_imgs}')
    print(f'   Already existed      : {skipped}')
    print(f'   Hand detection rate  : {found_rate:.1f}%')
    print(f'   No hand (black img)  : {no_hand}')

    if found_rate < 50:
        print('\n⚠️  WARNING: Low detection rate. Check dataset or lower confidence.')

    return out_dir

SKELETON_DIR = generate_skeleton_dataset(
    RAW_DIR, SKELETON_DIR,
    max_classes=MAX_CLASSES,
    max_per_class=MAX_IMGS_PER_CLS,
)

"""## Cell 8 — ✅ CLEANUP: Remove black/empty images"""

def cleanup_black_images(skeleton_dir):
    """Remove images where no hand was detected (black/empty)"""
    removed = 0
    kept = 0
    
    for class_dir in sorted(Path(skeleton_dir).iterdir()):
        if not class_dir.is_dir():
            continue
        
        for img_file in class_dir.glob("*.jpg"):
            img = cv2.imread(str(img_file))
            if img is None:
                os.remove(str(img_file))
                removed += 1
                continue
            
            # Check if mostly black (mean pixel value < 5)
            if np.mean(img) < 5:
                os.remove(str(img_file))
                removed += 1
            else:
                kept += 1
    
    print(f'\n✅ Cleanup complete!')
    print(f'   Kept    : {kept:,} valid skeleton images')
    print(f'   Removed : {removed:,} black/empty images')
    
    return kept, removed

cleanup_black_images(SKELETON_DIR)

"""## Cell 9 — Check class distribution"""

def check_class_distribution(skeleton_dir):
    print(f"\n{'Class':<15} {'Count':>8}")
    print("-" * 25)
    
    total = 0
    class_counts = {}
    
    for class_dir in sorted(Path(skeleton_dir).iterdir()):
        if not class_dir.is_dir():
            continue
        
        count = len(list(class_dir.glob("*.jpg")))
        class_counts[class_dir.name] = count
        total += count
        
        status = "⚠️ " if count < 500 else "✅"
        print(f"{status} {class_dir.name:<15} {count:>8,}")
    
    print("-" * 25)
    print(f"{'Total':<15} {total:>8,}")
    
    if class_counts:
        min_count = min(class_counts.values())
        max_count = max(class_counts.values())
        ratio = min_count / max_count if max_count > 0 else 0
        
        print(f"\nMin: {min_count:,}, Max: {max_count:,}, Balance ratio: {ratio:.2f}")
        
        if ratio < 0.5:
            print("⚠️  WARNING: Classes imbalanced! Model may favor larger classes.")
        else:
            print("✅ Classes reasonably balanced.")
    
    return class_counts

class_counts = check_class_distribution(SKELETON_DIR)

"""## Cell 10 — Visual verification"""

import random

def verify_skeletons(raw_dir, skeleton_dir, num_samples=10):
    import matplotlib.patches as mpatches
    
    classes = sorted([d for d in os.listdir(skeleton_dir)
                       if os.path.isdir(os.path.join(skeleton_dir, d))])
    chosen = random.sample(classes, min(num_samples, len(classes)))

    fig, axes = plt.subplots(len(chosen), 2, figsize=(9, 4 * len(chosen)))
    if len(chosen) == 1:
        axes = axes[np.newaxis, :]

    axes[0][0].set_title('Original Image', fontsize=13, fontweight='bold', pad=10)
    axes[0][1].set_title('Skeleton (model input)', fontsize=13, fontweight='bold', pad=10)

    all_ok = True
    detected = 0

    for row, cls in enumerate(chosen):
        imgs = os.listdir(os.path.join(skeleton_dir, cls))
        fname = random.choice(imgs)

        orig = cv2.imread(os.path.join(raw_dir, cls, fname))
        skel = cv2.imread(os.path.join(skeleton_dir, cls, fname))

        orig_rgb = cv2.cvtColor(cv2.resize(orig, (224, 224)), cv2.COLOR_BGR2RGB)
        skel_rgb = cv2.cvtColor(skel, cv2.COLOR_BGR2RGB)

        is_black = np.count_nonzero(skel_rgb) == 0
        if is_black:
            all_ok = False
        else:
            detected += 1

        status_text = '⚠️  No hand' if is_black else '✅  Hand detected'
        status_color = 'red' if is_black else '#00c875'

        axes[row][0].imshow(orig_rgb)
        axes[row][0].axis('off')
        axes[row][0].text(
            0.04, 0.96, cls.upper(),
            transform=axes[row][0].transAxes,
            fontsize=28, fontweight='black', color='white',
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='#1a1a2e',
                      edgecolor='white', linewidth=1.5, alpha=0.85)
        )

        axes[row][1].imshow(skel_rgb)
        axes[row][1].axis('off')
        axes[row][1].text(
            0.04, 0.96, cls.upper(),
            transform=axes[row][1].transAxes,
            fontsize=28, fontweight='black', color='white',
            va='top', ha='left',
            bbox=dict(boxstyle='round,pad=0.25', facecolor='#1a1a2e',
                      edgecolor=status_color, linewidth=2, alpha=0.9)
        )

    fig.suptitle(f'Skeleton Verification — {detected}/{len(chosen)} valid',
                 fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.show()

    if all_ok:
        print('✅ All samples valid — safe to train!')
    else:
        print('⚠️  Some images missing hands (should be minimal after cleanup)')

verify_skeletons(RAW_DIR, SKELETON_DIR, num_samples=10)

"""## Cell 11 — Train model (2-phase)"""

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Data generators
datagen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    width_shift_range=0.15,
    height_shift_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
)

train_gen = datagen.flow_from_directory(
    SKELETON_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='training', shuffle=True,
)
val_gen = datagen.flow_from_directory(
    SKELETON_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', subset='validation',
)

class_names = list(train_gen.class_indices.keys())
with open(CLASS_JSON_PATH, 'w') as f:
    json.dump(class_names, f)

print(f'Classes: {len(class_names)} → {class_names}')
print(f'Train  : {train_gen.samples} images')
print(f'Val    : {val_gen.samples} images')

# Build model
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False

x = GlobalAveragePooling2D()(base.output)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
out = Dense(len(class_names), activation='softmax')(x)

model = Model(inputs=base.input, outputs=out)
model.compile(optimizer=Adam(1e-3),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

print(f'\nTrainable params: {sum(tf.size(v).numpy() for v in model.trainable_variables):,}')

# Callbacks
callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3,
        min_lr=1e-6, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH, monitor='val_accuracy',
        save_best_only=True, verbose=1),
]

# Phase 1: Train with frozen base
print(f'\n🚀 PHASE 1: Training with frozen base (max {EPOCHS_PHASE1} epochs)\n')
history1 = model.fit(
    train_gen, epochs=EPOCHS_PHASE1,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
)

# Phase 2: Fine-tune
print(f'\n🚀 PHASE 2: Fine-tuning last 30 layers (max {EPOCHS_PHASE2} epochs)\n')
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=Adam(1e-5),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

history2 = model.fit(
    train_gen, epochs=EPOCHS_PHASE2,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
)

model.save(MODEL_FINAL_PATH)

best = max(history2.history['val_accuracy']) * 100
print(f'\n✅ Training complete — best val accuracy: {best:.2f}%')
print(f'   Best model : {MODEL_SAVE_PATH}')
print(f'   Final model: {MODEL_FINAL_PATH}')

"""## Cell 12 — Evaluate with confusion matrix"""

from sklearn.metrics import classification_report, confusion_matrix

# Load best model
best_model = tf.keras.models.load_model(MODEL_SAVE_PATH)

# Get predictions
val_gen.reset()
predictions = best_model.predict(val_gen, verbose=1)
y_pred = np.argmax(predictions, axis=1)
y_true = val_gen.classes[:len(y_pred)]

# Classification report
print('\n📊 Classification Report:\n')
print(classification_report(y_true, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.tight_layout()
plt.savefig('/content/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()
print('\nSaved: /content/confusion_matrix.png')

"""## Cell 13 — Plot training curves"""

# Combine both phases
combined_acc = history1.history['accuracy'] + history2.history['accuracy']
combined_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
combined_loss = history1.history['loss'] + history2.history['loss']
combined_val_loss = history1.history['val_loss'] + history2.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_total = range(1, len(combined_acc) + 1)
phase1_end = len(history1.history['accuracy'])

ax1.plot(epochs_total, combined_acc, 'b-o', label='Train', markersize=4)
ax1.plot(epochs_total, combined_val_acc, 'r-o', label='Validation', markersize=4)
ax1.axvline(phase1_end, color='gray', linestyle='--', label='Fine-tune starts', alpha=0.7)
ax1.set_title('Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(epochs_total, combined_loss, 'b-o', label='Train', markersize=4)
ax2.plot(epochs_total, combined_val_loss, 'r-o', label='Validation', markersize=4)
ax2.axvline(phase1_end, color='gray', linestyle='--', label='Fine-tune starts', alpha=0.7)
ax2.set_title('Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

mode_label = 'TEST MODE' if TEST_MODE else 'FULL TRAINING'
fig.suptitle(f'Training Curves [{mode_label}] — 2 Phases', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

"""## Cell 14 — Test on random samples"""

CONFIDENCE_THRESHOLD = 0.5

def predict_single(model, class_names, skeleton_float32):
    probs = model.predict(skeleton_float32[np.newaxis], verbose=0)[0]
    top_idx = int(np.argmax(probs))
    top_conf = float(probs[top_idx])
    label = class_names[top_idx] if top_conf >= CONFIDENCE_THRESHOLD else 'Uncertain'
    return label, top_conf

def test_random_samples(skeleton_dir, model, class_names, num_samples=12):
    classes = sorted([d for d in os.listdir(skeleton_dir)
                       if os.path.isdir(os.path.join(skeleton_dir, d))])
    chosen = random.sample(classes, min(num_samples, len(classes)))

    cols = 4
    rows = (len(chosen) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.5, rows * 4))
    axes = axes.flatten()

    correct = 0
    for i, cls in enumerate(chosen):
        imgs = os.listdir(os.path.join(skeleton_dir, cls))
        fname = random.choice(imgs)
        bgr = cv2.imread(os.path.join(skeleton_dir, cls, fname))
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        skel = cv2.resize(rgb, (224, 224)).astype('float32') / 255.0

        pred, conf = predict_single(model, class_names, skel)
        is_correct = (pred == cls)
        if is_correct: correct += 1

        axes[i].imshow(rgb)
        color = 'green' if is_correct else 'red'
        axes[i].set_title(
            f'True: {cls}\nPred: {pred} ({conf*100:.0f}%)',
            fontsize=11, color=color, fontweight='bold'
        )
        axes[i].axis('off')

    for j in range(len(chosen), len(axes)):
        axes[j].axis('off')

    accuracy = correct / len(chosen) * 100
    fig.suptitle(
        f'Sample Predictions — {correct}/{len(chosen)} correct ({accuracy:.0f}%)',
        fontsize=14, fontweight='bold'
    )
    plt.tight_layout()
    plt.savefig('/content/sample_predictions.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'\nSample accuracy: {correct}/{len(chosen)} ({accuracy:.0f}%)')

test_random_samples(SKELETON_DIR, best_model, class_names, num_samples=12)

"""## Cell 15 — Download all files"""

from google.colab import files

print('📥 Downloading trained model and results...')
files.download(MODEL_SAVE_PATH)
files.download(MODEL_FINAL_PATH)
files.download(CLASS_JSON_PATH)
files.download('/content/training_curves.png')
files.download('/content/confusion_matrix.png')
files.download('/content/sample_predictions.png')
print('✅ All files downloaded!')

"""## 🎉 DONE!

### Next steps:
1. Upload `sign_language_model_best.keras` and `class_names.json` to your FastAPI backend
2. Update FastAPI endpoint to use this model
3. Test with Angular frontend

### Expected results (FULL MODE):
- Validation accuracy: 90-97%
- Model size: ~14 MB
- Training time: ~35-45 min on T4 GPU
"""